# RSW / 04 — Burst reversion: enter against the burst when the regime switches back to calm (2021-01 – 2023-06)

Protocol: [`RSW 007`](../../discussions/MNQ_Regime_Switching/007_claude_rsw04_burst_reversion_preregistration.md). 1-minute fill simulation. Ticks per contract (1 tick = $0.50).

- **Entry**: at the first bar where the causal P(burst) ≤ 0.5 after a burst episode, against the episode's net move.
- **Exit**: at the time stop (the fit's mean calm duration + 5 bars), at a stop, or flat at 15:55 Chicago.
- **Strategies**: 2 regime models (M1 HMM, WB2 HSMM) × 5 stops (none, burst extreme + 2 ticks, 1.5σ̂, 3σ̂, 40 ticks) = 10.
- **Gates**:
  - **G1**: ≥ 150 trades.
  - **G2**: net > 0, Holm across 10 ≤ 0.05.
  - **G3**: net > 0 in ≥ 2 of 3 periods.
  - **G4**: beats the 95th percentile of the random-direction placebo (20 seeds).

In [ ]:
import json, os, sys, tempfile, pickle
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SYNTHETIC = os.environ.get("RSW_SYNTHETIC") == "1"
SYNTH_NULL = os.environ.get("RSW_SYNTH_NULL") == "1"
if SYNTHETIC:
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("RSW_SYNTH_DIR") or tempfile.mkdtemp(prefix="rsw_")
    os.environ["RSW_SYNTH_DIR"] = os.environ["EQUITY_TRADING_DIR"]

REPO = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(REPO / "src"))
from equity_trading import mnq_predict as P
from equity_trading import reversal as V
from equity_trading import regime_ar as R
from equity_trading import hsmm as HS
from equity_trading import burst_reversion as B
from equity_trading import trail_strategy as T
from equity_trading import micro_bars as MB

CFG = P.SYNTH_CFG if SYNTHETIC else P.MRPConfig()
RC, HC, BC = R.RSWConfig(), HS.HSMMConfig(), B.BRConfig()
D = dt.date
OUT = Path("04"); OUT.mkdir(exist_ok=True)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 50); pd.set_option("display.max_rows", 80)
print("engines", B.BR_VERSION, HS.HSMM_VERSION, R.RSW_VERSION, P.MRP_VERSION,
      ("| synthetic " + ("null" if SYNTH_NULL else "planted reversion")) if SYNTHETIC else "| real data",
      "| round trip", BC.round_trip_ticks, "ticks")
if SYNTHETIC and not (Path(os.environ["EQUITY_TRADING_DIR"]) / "Databento").exists():
    B.write_synthetic_burst_reversion(os.environ["EQUITY_TRADING_DIR"], CFG.warmup_start, CFG.train[1], CFG,
                                      revert_frac=0.0 if SYNTH_NULL else 0.6)

In [ ]:
if SYNTHETIC:
    FIRST_FIT, REFITS, END = (CFG.train[0], D(2020, 7, 1)), CFG.refits_train, CFG.train[1]
    PERIODS = {"2020Q3": (D(2020, 7, 1), D(2020, 9, 30)), "2020Q4": (D(2020, 10, 1), D(2020, 12, 31))}
else:
    FIRST_FIT, REFITS, END = (CFG.train[0], D(2021, 1, 1)), CFG.refits_train, CFG.train[1]
    PERIODS = {"2021": (D(2021, 1, 1), D(2021, 12, 31)), "2022": (D(2022, 1, 1), D(2022, 12, 31)),
               "2023H1": (D(2023, 1, 1), D(2023, 6, 30))}
NEED_PERIODS = int(np.ceil(2 / 3 * len(PERIODS)))
grid = P.load_grid(CFG.warmup_start, END, CFG)
sig2 = V.event_sigma(grid, FIRST_FIT, REFITS, END, CFG)
m1 = MB.load_1m(CFG.root, CFG.warmup_start - dt.timedelta(days=3), END + dt.timedelta(days=1))
S = R.build_series(grid, sig2, m1, CFG, RC)
S["iid"] = grid["instrument_id"].reindex(S.index).to_numpy()
S = S[(S["trade_date"] >= CFG.train[0]) & (S["trade_date"] <= END)]
M = T.minutes_from_1m(m1[m1.index >= pd.Timestamp(REFITS[0] - dt.timedelta(days=2), tz="UTC")],
                      T.TrailConfig(flat_ct=BC.flat_ct, commission_per_side=BC.commission_per_side))
del m1, grid
td = S["trade_date"].to_numpy()
print(f"{len(S):,} bars | {len(M.ts):,} minutes from {M.ts[0]}")

## 1. Walk-forward regime fits and burst probabilities

In [ ]:
cache = OUT / f"fits_{B.BR_VERSION}{'_synth' if SYNTHETIC else ''}{'_null' if SYNTH_NULL else ''}.pkl"
FITS = pickle.load(open(cache, "rb")) if cache.exists() else {}
for a in REFITS:
    fm = td < a
    if ("M1", str(a)) not in FITS:
        t0 = dt.datetime.now(); FITS[("M1", str(a))] = R.fit(S, 2, False, fm, RC)
        pickle.dump(FITS, open(cache, "wb")); print("M1 before", a, dt.datetime.now() - t0)
    if ("WB2", str(a)) not in FITS:
        t0 = dt.datetime.now(); FITS[("WB2", str(a))] = HS.fit(S, fm, FITS[("M1", str(a))], "WB", False, HC)
        pickle.dump(FITS, open(cache, "wb")); print("WB2 before", a, dt.datetime.now() - t0)
edges = list(REFITS) + [END + dt.timedelta(days=1)]
PB, HOLD, info = {}, {}, []
for name in ("M1", "WB2"):
    pb, hold = np.full(len(S), np.nan), np.zeros(len(S), int)
    for a, b in zip(edges[:-1], edges[1:]):
        f = FITS[(name, str(a))]
        seg = (td >= a) & (td < b)
        if name == "M1":
            p = R.filter_run(S, f)[[f"p{k}" for k in range(2)]].to_numpy()
            burst = int(np.argmax(f.s2)); calm = 1 - burst
            mean_calm = 1 / (1 - f.A[calm, calm])
        else:
            p = HS.filter_run(S, f, 2.0)[[f"p{k}" for k in range(2)]].to_numpy()
            burst = int(np.argmax(f.s2)); calm = 1 - burst
            mean_calm = f.mean_duration(calm)
        pb[seg] = p[seg, burst]
        hold[seg] = int(round(mean_calm)) + BC.extra_bars
        info.append({"model": name, "fit_before": str(a), "burst_sd": float(np.sqrt(f.s2[burst])), "calm_sd": float(np.sqrt(f.s2[calm])),
                     "burst_phi": float(f.phi[burst]), "calm_phi": float(f.phi[calm]), "mean_calm_bars": mean_calm,
                     "hold_bars": int(round(mean_calm)) + BC.extra_bars, "share_pb_gt_0.5": float(np.mean(pb[seg] > BC.p_burst))})
    PB[name], HOLD[name] = pb, hold
display(pd.DataFrame(info).round(4))

In [ ]:
EV = {name: B.episodes(S, PB[name], HOLD[name], BC) for name in ("M1", "WB2")}
for name, ev in EV.items():
    print(f"--- {name}: {len(ev)} episodes ending in a tradable switch | long (after down burst) {int((ev.side > 0).sum())}, "
          f"short {int((ev.side < 0).sum())}")
    display(ev[["burst_bars", "burst_move_ticks", "hold_bars"]].describe(percentiles=[.1, .5, .9]).round(1).T)

## 2. Strategies: 2 models × 5 stops, with placebo and mirror

In [ ]:
rows, TR = [], {}
for name, ev in EV.items():
    for kind in BC.stops:
        tr = B.simulate(M, ev, S.index, kind, BC)
        TR[(name, kind)] = tr
        st = B.stats_for(tr, PERIODS)
        st["skipped_session_end"] = tr.attrs.get("skipped_session_end", 0)
        plac = []
        for seed in range(BC.n_placebo):
            sides = np.random.default_rng(seed).choice([-1, 1], len(ev))
            pt = B.simulate(M, ev, S.index, kind, BC, sides=sides)
            plac.append(pt["net_ticks"].mean() if len(pt) else np.nan)
        mir = B.stats_for(B.simulate(M, ev, S.index, kind, BC, sides=-ev["side"].to_numpy()), PERIODS)
        rows.append({"model": name, "stop": kind, **st, "placebo_mean": float(np.nanmean(plac)),
                     "placebo_p95": float(np.nanquantile(plac, 0.95)), "mirror_net": mir["net"], "mirror_t": mir["t"]})
        print(name, kind, "trades", st["trades"], "net", round(st["net"], 2) if st["trades"] >= 3 else None,
              "| switches at/after 15:55 Chicago not traded:", st["skipped_session_end"])
G = pd.DataFrame(rows).set_index(["model", "stop"])
G["p_holm"] = pd.Series(R.holm({k: v for k, v in G["p"].fillna(1.0).items()}))
G["periods_pos"] = sum((G[f"net_{p}"] > 0).astype(int) for p in PERIODS)
G["G1"] = G["trades"] >= BC.min_trades
G["G2"] = (G["net"] > 0) & (G["p_holm"] <= BC.alpha)
G["G3"] = G["periods_pos"] >= NEED_PERIODS
G["G4"] = G["net"] > G["placebo_p95"]
G["pass"] = G[["G1", "G2", "G3", "G4"]].all(axis=1)
display(G.round(3).T)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4), sharey=True)
for i, name in enumerate(("M1", "WB2")):
    for kind in BC.stops:
        tr = TR[(name, kind)]
        if len(tr):
            ax[i].plot(pd.to_datetime(tr["entry_ts"]), tr["net_ticks"].cumsum(), lw=1, label=kind)
    ax[i].axhline(0, color="grey", lw=0.6); ax[i].set_title(f"{name}: cumulative net ticks per contract"); ax[i].legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "equity.png", dpi=110); plt.show()
# descriptive: net by burst size and by direction (time-stop-only rule)
for name in ("M1", "WB2"):
    tr = TR[(name, "none")]
    if len(tr) > 20:
        q = pd.qcut(tr["burst_move_ticks"].abs(), 4, duplicates="drop")
        print(f"--- {name}, no stop: net ticks by |burst move| quartile and by side")
        display(tr.groupby([q, "side"], observed=True)["net_ticks"].agg(["mean", "count"]).round(2).unstack())

## 3. Verdict

In [ ]:
passed = [f"{m}/{s}" for (m, s) in G.index[G["pass"]]]
verdict = ("CANDIDATE (2026 pre-registration): " + ", ".join(passed)) if passed else "NO_EDGE"
print(verdict)
summary = {"study": "RSW 04", "engine": B.BR_VERSION, "synthetic": SYNTHETIC, "synthetic_null": SYNTH_NULL,
           "round_trip_ticks": BC.round_trip_ticks, "fits": info,
           "episodes": {k: int(len(v)) for k, v in EV.items()},
           "results": json.loads(G.reset_index().to_json(orient="records")), "verdict": verdict}
json.dump(summary, open(OUT / "summary.json", "w"), indent=1, default=str)
print("written", OUT / "summary.json")